In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/sapporo_density.parquet')
is_unknown = (df["x"]==999) & (df["y"]==999)

df = df[~is_unknown].copy() 

# 6024088
print("Rows:", len(df))
print("d range:", df["d"].min(), df["d"].max())
print("Unique days:", df["d"].nunique())



Rows: 6024088
d range: 0 74
Unique days: 75


In [2]:
# 假設 d=0 是 2023-01-01
df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")

df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)


In [3]:
df = df.sort_values(["x","y","t","d"])

# 前一天
df["lag_1"] = df.groupby(["x","y","t"])["count"].shift(1)

# 前一週
df["lag_7"] = df.groupby(["x","y","t"])["count"].shift(7)

# 3日平均
df["rolling_3"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

# 7日平均
df["rolling_7"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df = df.dropna()

print("After lag rows:", len(df))


After lag rows: 4881569


In [4]:
max_day = df["d"].max()

TEST_DAYS = 7
VAL_DAYS = 7

test_start = max_day - TEST_DAYS + 1
val_start = test_start - VAL_DAYS

train_df = df[df["d"] <  val_start].copy()
val_df   = df[(df["d"] >= val_start) & (df["d"] < test_start)].copy()
test_df  = df[df["d"] >= test_start].copy()

print("Train days:", train_df["d"].min(), "~", train_df["d"].max())
print("Val days:", val_df["d"].min(), "~", val_df["d"].max(),)
print("Test days:", test_df["d"].min(), "~", test_df["d"].max())


Train days: 7 ~ 60
Val days: 61 ~ 67
Test days: 68 ~ 74


In [5]:
# baseline

# 前一天
pred_val_lag1  = val_df["lag_1"].values
pred_test_lag1 = test_df["lag_1"].values

#前七天
pred_val_lag7  = val_df["lag_7"].values
pred_test_lag7 = test_df["lag_7"].values


# 同時間同地點平均
key = ["weekday", "t", "x", "y"]
hist_mean = train_df.groupby(key)["count"].mean()

def predict_hist_mean(df_part: pd.DataFrame):
    idx = pd.MultiIndex.from_frame(df_part[key])
    return idx.map(hist_mean).fillna(0).to_numpy()

pred_val_hist  = predict_hist_mean(val_df)
pred_test_hist = predict_hist_mean(test_df)

In [6]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom)

def eval_metrics(name, y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "model": name,
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2":   r2_score(y_true, y_pred),
        "SMAPE": smape(y_true, y_pred),
    }

y_val  = val_df["count"].values
y_test = test_df["count"].values

results = []
# baselines on val
results += [eval_metrics("baseline_lag1 (val)",  y_val,  pred_val_lag1)]
results += [eval_metrics("baseline_lag7 (val)",  y_val,  pred_val_lag7)]
results += [eval_metrics("baseline_hist (val)",  y_val,  pred_val_hist)]

# baselines on test
results += [eval_metrics("baseline_lag1 (test)", y_test, pred_test_lag1)]
results += [eval_metrics("baseline_lag7 (test)", y_test, pred_test_lag7)]
results += [eval_metrics("baseline_hist (test)", y_test, pred_test_hist)]

pd.DataFrame(results).sort_values(["model"])

,model,MAE,RMSE,R2,SMAPE
5,baseline_hist (test),1.166909,1.740262,0.772630,0.465582
2,baseline_hist (val),1.160394,1.705420,0.778380,0.455963
3,baseline_lag1 (test),1.353030,2.367722,0.579113,0.428712
0,baseline_lag1 (val),1.377296,2.361393,0.575104,0.433097
4,baseline_lag7 (test),1.319972,2.101256,0.668517,0.430402
1,baseline_lag7 (val),1.398781,2.222935,0.623470,0.435828


In [7]:
import lightgbm as lgb
import joblib
from sklearn.metrics import r2_score

features = [
    "weekday", "t", "x", "y", "is_weekend",
    "lag_1", "lag_7", "rolling_3", "rolling_7"
]

train_sample = train_df.sample(n = min(800_000, len(train_df)), random_state=42)

X_train = train_sample[features]
y_train = train_sample["count"]

X_val = val_df[features]
y_val = val_df['count']

X_test = test_df[features]
y_test = test_df["count"]

model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=4
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(50)]
)

pred_val  = model.predict(X_val)
pred_test = model.predict(X_test)

print(eval_metrics("LGBM (val)",  y_val,  pred_val))
print(eval_metrics("LGBM (test)", y_test, pred_test))

joblib.dump(model, "lgbm_sapporo_2.pkl")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011696 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1045
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 9
[LightGBM] [Info] Start training from score 3.287492
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[60]	valid_0's rmse: 1.63025	valid_0's l2: 2.65772
{'model': 'LGBM (val)', 'MAE': 1.132758042396866, 'RMSE': 1.6302514295888562, 'R2': 0.7974858191682556, 'SMAPE': np.float64(0.417652731197564)}
{'model': 'LGBM (test)', 'MAE': 1.114515074604249, 'RMSE': 1.6719147559792444, 'R2': 0.790138824777249, 'SMAPE': np.float64(0.41528657849484485)}


['lgbm_sapporo_2.pkl']